## RAG(检索增强生成) ——文本分块（Chunking）与向量嵌入（Embedding）机制
1. Embedding 的数学本质与对齐：理解离散的自然语言如何被压缩并映射到高维连续向量空间，以及向量相似度（余弦相似度）的计算流。
2. 高级分块策略（Recursive Character Chunking）：丢弃幼稚的固定长度切片，手写一个基于语义层级（段落、句子、词）的递归文本切分器，解决 RAG 召回阶段的“上下文断裂”痛点。

#### 为什么大模型需要 RAG？
尽管模型的上下文窗口（Context Window）越来越大，但直接将几百页的文档全部塞进 Prompt 存在三个致命问题:
* 成本昂贵：每一次对话都要为几百万 Token 的上下文付费。
* Lost in the Middle（长上下文迷失）：LLM 容易忽略超长文本中间部分的信息。
* 时效性与知识死角：预训练数据的知识是静态的，大模型无法得知企业私有数据或昨日的新闻。

RAG 的核心思想是“开卷考试”：当用户提问时，系统先去海量私有文档库中检索（Retrieval）出最相关的几段话，然后再把这几段话和问题一起喂给 LLM 生成（Generation）回答。

#### Embedding（向量嵌入）的本质
如何让计算机知道“国王”和“女王”很接近，而“国王”和“香蕉”很远？这就是 Embedding 的工作。
Embedding 是一个深度学习模型（如 `text-embedding-3-small`），它将一段任意长度的文本，映射为一个固定维度的实数向量（例如 1536 维）.

在这个高维空间中，语义相似的文本，其向量的几何距离（或夹角）就越近。
我们通常使用余弦相似度（Cosine Similarity）来衡量两段文本的语义接近程度。其数学公式为：
$$\text{Similarity} = \cos(\theta) = \frac{A \cdot B}{\Vert{}A\Vert{} \Vert{}B\Vert{}}$$
结果越接近 1，代表语义越相似。

#### Chunking（文本分块）的工程艺术
在将文档转化为向量前，必须将长文档切成“块”（Chunks）。如果切得不好，RAG 的性能会雪崩：
* 固定长度切片（Fixed-size Chunking）：比如每 100 个字切一块。这会导致一句话被从中间斩断，语义直接丢失。
* 递归字符切片（Recursive Character Chunking）：这是工业界标准的做法。它使用一个分隔符列表（如 `["\n\n", "\n", "。", "，", ""]`），优先按段落切，如果段落太大，再按句子切，直到满足设定的 Chunk Size，从而最大程度保留了语义的完整性。

**纯 Python 手写一个递归文本切分器，并模拟一次 RAG 的 Embedding 召回过程**


In [ ]:
import re
import numpy as np
from typing import List

# --- 1. 手写递归字符分块器 (Recursive Character Splitter) ---
class MiniRecursiveCharacterTextSplitter:
    """精简版递归字符文本切分器"""
    def __init__(self, chunk_size: int = 100, chunk_overlap: int = 20):
        self.chunk_size = chunk_size        # 每个块的最大字符数
        self.chunk_overlap = chunk_overlap  # 块与块之间的重叠字数（防止边界语义断裂）
        self.separators = ["\n\n", "\n", "。", "！", "？", "，", " ", ""]

    def split_text(self, text: str) -> List[str]:
        return self._split_recursive(text, self.separators)

    def _split_recursive(self, text: str, separators: List[str]) -> List[str]:
        # 如果文本已经足够小，直接返回
        if len(text) <= self.chunk_size:
            return [text]

        # 如果没有可选的分隔符了，强行截断
        if not separators:
            return [text[i:i + self.chunk_size] for i in range(0, len(text), self.chunk_size - self.chunk_overlap)]

        # 选择当前层级的最佳分隔符
        current_sep = separators[0]
        next_seps = separators[1:]

        # 将文本按照当前分隔符切开
        if current_sep == "":
            splits = list(text)
        else:
            # 保留分隔符的切分逻辑
            splits = text.split(current_sep)
            # 把分隔符补回去，保持可读性
            splits = [s + current_sep for s in splits[:-1]] + [splits[-1]]
            splits = [s for s in splits if s] # 过滤空字符串

        chunks = []
        current_chunk = ""

        for split in splits:
            # 如果加上这个切片没有超限，就继续累加
            if len(current_chunk) + len(split) <= self.chunk_size:
                current_chunk += split
            else:
                # 现有累加已经够大，存入 chunks
                if current_chunk:
                    chunks.append(current_chunk)

                # 如果单个 split 就已经超过了限制，递归交给下一级分隔符细切
                if len(split) > self.chunk_size:
                    chunks.extend(self._split_recursive(split, next_seps))
                    current_chunk = ""
                else:
                    # 利用 overlap 机制，将上一个块的末尾部分继承过来
                    overlap_start = max(0, len(current_chunk) - self.chunk_overlap)
                    current_chunk = current_chunk[overlap_start:] + split

        if current_chunk:
            chunks.append(current_chunk)

        return chunks


# --- 2. 模拟真实 Embedding 空间 ---
# 工业界会调用 OpenAI/HuggingFace 的 Embedding 接口，这里我们用确定性的随机高维向量模拟语义空间
def get_mock_embedding(text: str) -> np.ndarray:
    """根据文本特征生成模拟的 256 维向量"""
    np.random.seed(abs(hash(text)) % (10 ** 8))
    vec = np.random.randn(256)

    # 模拟语义对齐：如果文本包含某些高频词，手动扭转维度以模拟真实的“语义接近”
    if "Agent" in text or "智能体" in text:
        vec[0:10] += 2.0
    if "大模型" in text or "LLM" in text:
        vec[10:20] += 2.0
    if "防爆" in text or "隐形炸弹" in text:
        vec[50:60] += 3.0

    # 归一化（L2范数归一化，方便直接计算点积作为余弦相似度）
    return vec / np.linalg.norm(vec)

def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    """计算两个已归一化向量的余弦相似度"""
    return float(np.dot(v1, v2))


# --- 3. 运行端到端检索流水线 ---
if __name__ == "__main__":
    # 模拟一份企业本地私有文档（来自第三周的 Agent 讲义）
    raw_document = """
在大模型的工业界落地中，有三大“隐形炸弹”会导致 Agent 频繁挂掉。
第一是结构化输出的脆弱性，大模型生成格式易损坏。
第二是观察值暴涨，比如搜索引擎返回的Observation过长会导致显存爆炸（OOM）。
第三是规划失控，模型容易陷入无意义的死循环。
为了解决这些问题，我们需要在控制流中加入容错解析器和滑动窗口记忆管理器，从而打造生产级高可用的智能体引擎。
    """

    print("📝 [步骤 1] 正在进行高级递归文本分块...")
    splitter = MiniRecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=15)
    doc_chunks = splitter.split_text(raw_document)

    for idx, chunk in enumerate(doc_chunks):
        print(f"  Chunk [{idx}]: (长度:{len(chunk)}) -> {chunk.strip()}")

    print("\n🔮 [步骤 2] 将文本块转化为高维语义向量 (Embedding)...")
    chunk_embeddings = [get_mock_embedding(chunk) for chunk in doc_chunks]
    print(f"  成功将 {len(doc_chunks)} 个文本块转化为 256 维稠密向量。")

    # 模拟用户提问
    user_query = "Agent 长文本导致显存爆炸怎么办？"
    print(f"\n🔍 [步骤 3] 用户提问: '{user_query}'，计算查询向量...")
    query_embedding = get_mock_embedding(user_query)

    print("\n🎯 [步骤 4] 语义相似度匹配 (Vector Search)...")
    search_results = []
    for idx, chunk_emb in enumerate(chunk_embeddings):
        score = cosine_similarity(query_embedding, chunk_embeddings[idx])
        search_results.append((score, doc_chunks[idx]))

    # 按相似度从高到低排序
    search_results.sort(key=lambda x: x[0], reverse=True)

    for rank, (score, chunk_text) in enumerate(search_results[:2]):
        print(f"  Top {rank+1} [得分: {score:.4f}]:\n  >>> {chunk_text.strip()}")

1. 观察重叠区（Overlap）的作用：
    * 运行上述代码，观察输出的各个 `Chunk` 的交界处。
    * 思考：为什么在切分代码中我们要刻意让上一个块的末尾与下一个块的开头保留 `chunk_overlap=15` 个字符的交集？如果将 `chunk_overlap` 设为 0，对 RAG 的召回率（Recall）可能会造成什么毁灭性影响？

2. 多语言切分的隐性 Bug：
    * 上述代码中我们定义的分隔符 `separators = ["\n\n", "\n", "。", "！", "？", "，", " ", ""]` 主要是针对中文符号的。
    * 工程挑战：如果是中英文混合的中大型英文文档，由于英文是以单词（Word）为单位，且句号是英文半角`.`，直接使用这套切分器会发生什么情况？你将如何优化 `separators` 列表，使得切分器能完美兼顾英文的单词边界，而不至于把 `Agent` 斩断成 `Ag` 和 `ent`？

## RAG 核心基石——Embedding（向量嵌入）
Embedding（向量嵌入）是RAG 检索系统的底层心脏。

#### 为什么需要 Embedding？（从文本到数值的桥梁）
1. 计算机的“语言盲区”：
    * 计算机本身无法直接理解“国王”、“女王”或“苹果”等自然语言符号的内在含义。传统的搜索技术（如 BM25、倒排索引）依赖于精确词匹配（Keyword Matching）。
        * 局限性：如果用户搜索“如何修复手机卡顿”，而文档中写的是“解决智能手机运行缓慢”，传统搜索会因为词汇不匹配而无法精准召回。
2. Embedding 的核心本质：
    * Embedding 是一种将离散的自然语言文本（Token、句子、段落）映射到高维连续向量空间（如 1536 维、3072 维）的深度学习模型 。
        * 语义几何化：在这个高维空间中，语义越接近的文本，其向量在空间中的几何距离就越近 。
        * 经典示例：$\text{Vector("国王")} - \text{Vector("男人")} + \text{Vector("女人")} \approx \text{Vector("女王")}$ 。

#### 语义相似度的计算方法
在向量空间中，评估两段文本语义是否相似，核心是计算它们对应向量之间的距离或夹角。常用方法包括：
1. 余弦相似度（Cosine Similarity）—— 最推荐/最常用
    * 衡量两个向量方向上的夹角余弦值，取值范围为 $[-1, 1]$（越接近 1 说明语义越接近）。
        * 数学公式：
        $$\text{Similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\Vert{}\mathbf{A}\Vert{} [cite_start]\Vert{}\mathbf{B}\Vert{}} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$
        * 特点：对向量的长度/模长不敏感，只关注方向。因此在处理长短不一的文本嵌入时稳定性最高。

2. 点积/内积（Dot Product）
    * 数学公式：$\text{DotProduct}(\mathbf{A}, \mathbf{B}) = \mathbf{A} \cdot \mathbf{B} = \sum_{i=1}^{n} A_i B_i$
    * 特点：计算速度极快。若 Embedding 向量已经过单位化（L2 归一化，模长 $\Vert{}\mathbf{A}\Vert{}=1$），则点积结果完全等价于余弦相似度。

3. 欧氏距离（Euclidean Distance / L2 Distance）
    * 数学公式：$d(\mathbf{A}, \mathbf{B}) = \sqrt{\sum_{i=1}^{n} (A_i - B_i)^2}$
    * 特点：测量几何空间中的绝对直线距离，距离越小说明越相似。

#### 主流 Embedding 模型与选型
在工程落地中，选择合适的 Embedding 模型对 RAG 检索召回率至关重要：
1. 云端 API 模型：
    * OpenAI `text-embedding-3-small` / `text-embedding-3-large`：成本低、性能优异，支持动态维度裁剪（Matryoshka Representation Learning）。
    * Cohere Embed v3：对多语言及混合检索（Dense + Sparse）支持极佳。

2. 开源/本地部署模型（参考 MTEB / C-MTEB 榜单）：
    * BGE 系列（BAAI 智源研究院）：如 `bge-large-zh-v1.5`，中文语义表现极其强劲，是目前国内企业落地首选之一。
    * BGE-M3：支持多语言（Multi-linguality）、多粒度（Multi-granularity）、多检索模式（Multi-capability）。
    * Jina Embedding v2/v3：支持 8k 超长上下文嵌入，适合长文本直接向量化。


以下是基于Python代码计算文本语义相似度（使用开源模型进行本地语义向量抽取与相似度计算）：

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import numpy as np
from sentence_transformers import SentenceTransformer

# 1. 定义余弦相似度计算函数
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 2. 加载本地开源 Embedding 模型（以智源 BGE 模型为例）
model = SentenceTransformer('BAAI/bge-small-zh-v1.5')

# 3. 准备测试文本
query = "大模型怎么微调？"
doc1 = "使用 LoRA 进行高效参数微调的步骤与原理"
doc2 = "今天天气很好，适合出去户外散步"

# 4. 生成向量 (Embedding)
emb_query = model.encode(query)
emb_doc1 = model.encode(doc1)
emb_doc2 = model.encode(doc2)

# 5. 计算并打印相似度
sim_1 = cosine_similarity(emb_query, emb_doc1)
sim_2 = cosine_similarity(emb_query, emb_doc2)

print(f"Query 与 Doc1 相似度: {sim_1:.4f}")  # 输出得分通常高于 0.7~0.8
print(f"Query 与 Doc2 相似度: {sim_2:.4f}")  # 输出得分通常低于 0.2~0.3

1. 思考题：在 RAG 场景中，为什么直接用用户的“疑问句”（Query）去检索知识库中的“陈述句”（Chunk）有时匹配度不够高？（提示：问句与答句在文本表达形态上属于非对称检索）。
2. 动手实践：安装 `sentence-transformers` 库，加载一个轻量级 Embedding 模型，尝试输入 3 段不同主题的文本，观察并打印它们两两之间的余弦相似度。